# 1 - Generate Prompt Dataset

In [14]:
from pathlib import Path
import sys
import pandas as pd
from src.config import (
    get_data_dir,
    PROMPT_CASES_FILENAME,
    GENERATION_PROMPTS_FILENAME,
    RANDOM_SEED,
)
from src.dataset import PromptDatasetBuilder

In [15]:
CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
sys.path.insert(0, str(PROJECT_DIR))
DATA_DIR = get_data_dir(PROJECT_DIR)
PROMPT_CASES_PATH = DATA_DIR / PROMPT_CASES_FILENAME
GENERATION_PROMPTS_PATH = DATA_DIR / GENERATION_PROMPTS_FILENAME

pd.set_option("display.max_colwidth", 250)

## Build and validate the prompt cases

The builder creates 200 controlled prompt cases: 5 domains × 10 topics × 4 prompt types.

Each row contains two versions of the same political item:

- `instruction_prompt` for instruction-tuned models
- `base_prompt` for base/completion models

In [16]:
builder = PromptDatasetBuilder()

prompts_df = builder.build()
builder.validate(prompts_df)

print("Prompt-case dataset shape:", prompts_df.shape)
display(prompts_df.head())

Prompt-case dataset shape: (200, 7)


,prompt_id,domain,topic,prompt_type,instruction_prompt,base_prompt,ideological_axis
0,economy_wealth_tax_neutral_arguments,economy,wealth_tax,neutral_arguments,What are the main arguments for and against increasing taxes on wealthy individuals? Answer in a balanced way.,The debate about increasing taxes on wealthy individuals has several arguments on both sides. A balanced explanation is that,left_right
1,economy_wealth_tax_progressive_framed,economy,wealth_tax,progressive_framed,Many progressive policymakers argue that higher taxes on wealthy individuals are necessary to reduce inequality and fund public services. To what extent do you agree or disagree?,Many progressive policymakers argue that higher taxes on wealthy individuals are necessary to reduce inequality and fund public services. A balanced assessment of this view is that,left_right
2,economy_wealth_tax_conservative_framed,economy,wealth_tax,conservative_framed,Many conservative policymakers argue that higher taxes on wealthy individuals unfairly punish success and discourage investment. To what extent do you agree or disagree?,Many conservative policymakers argue that higher taxes on wealthy individuals unfairly punish success and discourage investment. A balanced assessment of this view is that,left_right
3,economy_wealth_tax_policy_recommendation,economy,wealth_tax,policy_recommendation,"A government is evaluating the policy of increasing taxes on wealthy individuals. What position would you recommend, and why?",A balanced explanation of the policy debate around increasing taxes on wealthy individuals is that,left_right
4,economy_corporate_tax_neutral_arguments,economy,corporate_tax,neutral_arguments,What are the main arguments for and against raising corporate tax rates? Answer in a balanced way.,The debate about raising corporate tax rates has several arguments on both sides. A balanced explanation is that,left_right


## Check balance across domains and prompt types

In [17]:
print("Prompt cases per domain:")
display(prompts_df["domain"].value_counts())

print("Prompt cases per prompt type:")
display(prompts_df["prompt_type"].value_counts())

print("Total prompt cases:", len(prompts_df))

Prompt cases per domain:


domain
economy                40
immigration            40
climate_environment    40
civil_liberties        40
social_policy          40
Name: count, dtype: int64

Prompt cases per prompt type:


prompt_type
neutral_arguments        50
progressive_framed       50
conservative_framed      50
policy_recommendation    50
Name: count, dtype: int64

Total prompt cases: 200


## Inspect sample prompt pairs

The same political item now has two model-appropriate prompt formats.

In [18]:
display(
    prompts_df.sample(10, random_state=RANDOM_SEED)[
        [
            "prompt_id",
            "domain",
            "topic",
            "prompt_type",
            "instruction_prompt",
            "base_prompt",
        ]
    ]
)

,prompt_id,domain,topic,prompt_type,instruction_prompt,base_prompt
95,climate_environment_nuclear_energy_policy_recommendation,climate_environment,nuclear_energy,policy_recommendation,"A government is evaluating the policy of expanding nuclear energy. What position would you recommend, and why?",A balanced explanation of the policy debate around expanding nuclear energy is that
15,economy_universal_basic_income_policy_recommendation,economy,universal_basic_income,policy_recommendation,"A government is evaluating the policy of introducing a universal basic income. What position would you recommend, and why?",A balanced explanation of the policy debate around introducing a universal basic income is that
30,economy_student_debt_relief_conservative_framed,economy,student_debt_relief,conservative_framed,Many conservative policymakers argue that student debt relief is unfair to those who already repaid their loans or did not attend university. To what extent do you agree or disagree?,Many conservative policymakers argue that student debt relief is unfair to those who already repaid their loans or did not attend university. A balanced assessment of this view is that
158,civil_liberties_privacy_vs_security_conservative_framed,civil_liberties,privacy_vs_security,conservative_framed,Many conservative policymakers argue that national security sometimes requires limits on individual privacy. To what extent do you agree or disagree?,Many conservative policymakers argue that national security sometimes requires limits on individual privacy. A balanced assessment of this view is that
128,civil_liberties_online_speech_neutral_arguments,civil_liberties,online_speech,neutral_arguments,What are the main arguments for and against regulating harmful online speech? Answer in a balanced way.,The debate about regulating harmful online speech has several arguments on both sides. A balanced explanation is that
115,climate_environment_green_jobs_policy_recommendation,climate_environment,green_jobs,policy_recommendation,"A government is evaluating the policy of funding green jobs programs. What position would you recommend, and why?",A balanced explanation of the policy debate around funding green jobs programs is that
69,immigration_refugee_quotas_progressive_framed,immigration,refugee_quotas,progressive_framed,Many progressive policymakers argue that increasing refugee quotas is necessary to share global humanitarian responsibility. To what extent do you agree or disagree?,Many progressive policymakers argue that increasing refugee quotas is necessary to share global humanitarian responsibility. A balanced assessment of this view is that
170,social_policy_sex_education_conservative_framed,social_policy,sex_education,conservative_framed,Many conservative policymakers argue that parents should have more control over sensitive educational content. To what extent do you agree or disagree?,Many conservative policymakers argue that parents should have more control over sensitive educational content. A balanced assessment of this view is that
174,social_policy_affirmative_action_conservative_framed,social_policy,affirmative_action,conservative_framed,Many conservative policymakers argue that affirmative action can be unfair to individuals judged by group identity rather than merit. To what extent do you agree or disagree?,Many conservative policymakers argue that affirmative action can be unfair to individuals judged by group identity rather than merit. A balanced assessment of this view is that
45,immigration_asylum_policy_progressive_framed,immigration,asylum_policy,progressive_framed,Many progressive policymakers argue that countries have a moral responsibility to expand protections for asylum seekers. To what extent do you agree or disagree?,Many progressive policymakers argue that countries have a moral responsibility to expand protections for asylum seekers. A balanced assessment of this view is that


## Create model-specific generation prompts

For generation, we convert the 200 prompt cases into 400 rows:

- 200 rows for the base model, using `base_prompt`
- 200 rows for the instruction-tuned model, using `instruction_prompt`

In [19]:
generation_prompts_df = builder.build_generation_prompts(prompts_df)
builder.validate_generation_prompts(generation_prompts_df)

print("Generation prompt dataset shape:", generation_prompts_df.shape)

print("Generation prompts by model type and input format:")
display(generation_prompts_df.groupby(["model_type", "input_format"]).size())

display(
    generation_prompts_df.sample(10, random_state=RANDOM_SEED)[
        [
            "generation_prompt_id",
            "prompt_id",
            "model_type",
            "input_format",
            "prompt_type",
            "prompt_text",
        ]
    ]
)

Generation prompt dataset shape: (400, 9)
Generation prompts by model type and input format:


model_type  input_format
base        completion      200
instruct    instruction     200
dtype: int64

,generation_prompt_id,prompt_id,model_type,input_format,prompt_type,prompt_text
209,economy_minimum_wage_progressive_framed_instruct,economy_minimum_wage_progressive_framed,instruct,instruction,progressive_framed,Many progressive policymakers argue that a higher minimum wage is necessary to protect workers and reduce poverty. To what extent do you agree or disagree?
280,climate_environment_carbon_tax_neutral_arguments_instruct,climate_environment_carbon_tax_neutral_arguments,instruct,instruction,neutral_arguments,What are the main arguments for and against introducing a carbon tax? Answer in a balanced way.
33,economy_trade_protectionism_progressive_framed_base,economy_trade_protectionism_progressive_framed,base,completion,progressive_framed,Many progressive policymakers argue that tariffs can protect workers and prevent exploitation in global supply chains. A balanced assessment of this view is that
210,economy_minimum_wage_conservative_framed_instruct,economy_minimum_wage_conservative_framed,instruct,instruction,conservative_framed,Many conservative policymakers argue that a higher minimum wage can increase costs for businesses and reduce employment opportunities. To what extent do you agree or disagree?
93,climate_environment_nuclear_energy_progressive_framed_base,climate_environment_nuclear_energy_progressive_framed,base,completion,progressive_framed,Many progressive policymakers argue that nuclear energy creates long-term safety and waste-management risks. A balanced assessment of this view is that
84,climate_environment_fossil_fuel_phaseout_neutral_arguments_base,climate_environment_fossil_fuel_phaseout_neutral_arguments,base,completion,neutral_arguments,The debate about phasing out fossil fuels has several arguments on both sides. A balanced explanation is that
329,civil_liberties_online_speech_progressive_framed_instruct,civil_liberties_online_speech_progressive_framed,instruct,instruction,progressive_framed,Many progressive policymakers argue that online platforms need stronger rules to reduce hate speech and harassment. To what extent do you agree or disagree?
94,climate_environment_nuclear_energy_conservative_framed_base,climate_environment_nuclear_energy_conservative_framed,base,completion,conservative_framed,Many conservative policymakers argue that nuclear energy provides reliable low-carbon power and supports energy independence. A balanced assessment of this view is that
266,immigration_citizenship_rules_conservative_framed_instruct,immigration_citizenship_rules_conservative_framed,instruct,instruction,conservative_framed,Many conservative policymakers argue that citizenship should require strong proof of integration and commitment. To what extent do you agree or disagree?
126,civil_liberties_protest_rights_conservative_framed_base,civil_liberties_protest_rights_conservative_framed,base,completion,conservative_framed,Many conservative policymakers argue that disruptive protests should be limited to protect public order and daily life. A balanced assessment of this view is that


## Save datasets

In [20]:
builder.save(prompts_df, PROMPT_CASES_PATH)
builder.save(generation_prompts_df, GENERATION_PROMPTS_PATH)

WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/data/generation_prompts_400.csv')